# Pipeline Pick & Place por Color — MyCobot 280

**P6 + P7 — Visión + Integración End-to-End**
**Curso:** ROB 2026

## Flujo del programa

1. El robot se mueve a la **posición inicial** (donde la cámara observa la zona del cubo).
2. La cámara captura una imagen, mide su **ancho y alto**, detecta el **color del cubo** y su **posición en píxeles**.
3. El robot **ajusta X/Y** sobre la posición inicial para alinearse con el cubo.
4. El robot **baja, agarra** el cubo y lo **sube**.
5. Según el color detectado, el robot lleva el cubo a su **zona correspondiente** (rojo, azul, verde o amarillo).
6. Suelta el cubo y vuelve a la posición inicial.

## Diagrama de estados

```
Inicio
  ↓
Posición de observación
  ↓
Detectar color del cubo
  ↓
Conversión píxel a mm
  ↓
Calcular posición de agarre
  ↓
Agarrar cubo
  ↓
¿Qué color es?
  ↓        ↓        ↓        ↓
ROJO    VERDE    MORADO   AMARILLO
  ↓        ↓        ↓        ↓
Fin
```


## 1. Importaciones y Conexión

Se importa la librería `pymycobot` para controlar el robot, `cv2` (OpenCV) para visión, `numpy` para operaciones de matrices y `time` para sincronización.


In [ ]:
from pymycobot.mycobot import MyCobot
import cv2
import numpy as np
import time

# ============================================================
# CONEXIÓN AL ROBOT Y CÁMARA
# ============================================================
mc = MyCobot('/dev/ttyUSB0', 1000000)
mc.power_on()
time.sleep(1)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
time.sleep(1)

print("Robot conectado")
print(f"Ángulos: {mc.get_angles()}")
print(f"Coords:  {mc.get_coords()}")

## 2. Posiciones del Robot

### Posición inicial / observación
Es donde el robot se queda mirando hacia abajo para detectar el cubo con la cámara.

| Parámetro | Valor |
|---|---|
| Ángulos | `[20.65, -6.76, 15.2, -85.07, 0.08, -24.52]` |
| Coordenadas | `[98.8, -31.7, 305.4, -167.77, -5.42, -45.39]` |

### Posiciones fijas por color
Cada color tiene una zona de depósito fija medida físicamente.

| Color | Ángulos articulares |
|---|---|
| ROJO | `[80.59, -40.78, -89.03, 51.15, 4.48, -62.31]` |
| AZUL | `[114.87, -33.75, -97.91, 50.97, 4.65, -29.7]` |
| VERDE | `[98.96, -30.41, -101.86, 50.88, 3.6, -29.61]` |
| AMARILLO | `[66.97, -42.89, -75.32, 38.84, 3.95, -60.55]` |


In [ ]:
# ============================================================
# POSICIÓN INICIAL (donde la cámara observa el cubo)
# ============================================================
pose_inicial = [20.65, -6.76, 15.2, -85.07, 0.08, -24.52]

# Coordenadas de la pose inicial
COORDS_INICIAL = [98.8, -31.7, 305.4, -167.77, -5.42, -45.39]

# Altura a la que va a bajar para agarrar (Z en mm)
Z_AGARRE = 130   # ajustar según la altura real del cubo en la mesa
Z_ALTO   = 250   # altura segura para moverse sin chocar

# ============================================================
# POSICIONES FIJAS POR COLOR
# ============================================================
POSICIONES_POR_COLOR = {
    "ROJO":     [80.59,  -40.78, -89.03,  51.15, 4.48, -62.31],
    "AZUL":     [114.87, -33.75, -97.91,  50.97, 4.65, -29.7],
    "VERDE":    [98.96,  -30.41, -101.86, 50.88, 3.6,  -29.61],
    "AMARILLO": [66.97,  -42.89, -75.32,  38.84, 3.95, -60.55],
}

VELOCIDAD = 25

print("Posiciones cargadas correctamente")
for color, pose in POSICIONES_POR_COLOR.items():
    print(f"  {color:10s} -> {pose}")

## 3. Calibración Cámara → Robot y Rangos HSV

### Calibración píxel → mm
`MM_PER_PIXEL` indica cuántos milímetros equivale 1 píxel de la imagen.
Para calibrarlo: poner una regla sobre la mesa y medir cuántos píxeles ocupan 100 mm.

### Rangos HSV
Los rangos de color están calibrados para las condiciones del laboratorio.
El rojo tiene dos rangos porque envuelve el círculo cromático HSV.


In [ ]:
# ============================================================
# CALIBRACIÓN CÁMARA → ROBOT
# ============================================================
# Cuántos mm equivale 1 píxel de la imagen
MM_PER_PIXEL = 0.5   # AJUSTAR con la calibración real

# ============================================================
# RANGOS HSV PARA DETECCIÓN DE COLOR
# ============================================================
RANGOS_HSV = {
    "ROJO": [
        (np.array([0, 120, 70]),   np.array([10, 255, 255])),
        (np.array([170, 120, 70]), np.array([180, 255, 255])),
    ],
    "AZUL":     [(np.array([100, 120, 70]),  np.array([130, 255, 255]))],
    "VERDE":    [(np.array([40, 80, 60]),    np.array([85, 255, 255]))],
    "AMARILLO": [(np.array([20, 100, 100]),  np.array([35, 255, 255]))],
}

MIN_AREA = 1200   # área mínima en píxeles para validar la detección

print(f"MM_PER_PIXEL = {MM_PER_PIXEL}")
print(f"MIN_AREA     = {MIN_AREA} px²")
print(f"Colores configurados: {list(RANGOS_HSV.keys())}")

## 4. Función: Detectar Color y Medir Cubo

Esta función:

- Captura una imagen desde la cámara.
- Imprime el **ancho y alto** de la imagen.
- Recorre los 4 colores definidos y busca contornos.
- Para cada color encontrado imprime: **área, centro, tamaño en píxeles**.
- Selecciona el color con mayor área como el detectado.
- Calcula el **desplazamiento en milímetros** del cubo respecto al centro de la imagen.
- Guarda una imagen anotada en `/tmp/cubo_detectado.jpg`.


In [ ]:
def detectar_color_y_medir():
    """
    Captura una imagen, detecta el color del cubo y mide su posición
    respecto al centro de la imagen. Devuelve un diccionario con la info.
    """
    # Limpiar buffer de la cámara
    for _ in range(5):
        cap.read()

    ret, frame = cap.read()
    if not ret:
        print("ERROR: no se pudo capturar imagen")
        return None

    alto, ancho, _ = frame.shape
    cx_img, cy_img = ancho // 2, alto // 2

    print(f"\n========== INFORMACIÓN DE LA IMAGEN ==========")
    print(f"Ancho de imagen:  {ancho} px")
    print(f"Alto de imagen:   {alto} px")
    print(f"Centro imagen:    ({cx_img}, {cy_img})")

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    kernel = np.ones((5, 5), np.uint8)

    mejor_color = None
    mejor_area = 0
    mejor_cx, mejor_cy = 0, 0
    mejor_w, mejor_h = 0, 0

    print(f"\n========== DETECCIÓN DE COLORES ==========")

    for color, rangos in RANGOS_HSV.items():
        # Construir máscara (rojo tiene 2 rangos, se unen)
        mascara = None
        for lo, hi in rangos:
            m = cv2.inRange(hsv, lo, hi)
            mascara = m if mascara is None else cv2.bitwise_or(mascara, m)

        # Limpieza morfológica
        mascara = cv2.erode(mascara, kernel, iterations=1)
        mascara = cv2.dilate(mascara, kernel, iterations=2)

        # Buscar contornos
        contornos, _ = cv2.findContours(
            mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        area_max = 0
        cx_c, cy_c, w_c, h_c = 0, 0, 0, 0

        for c in contornos:
            area = cv2.contourArea(c)
            if area < MIN_AREA:
                continue

            M = cv2.moments(c)
            if M["m00"] == 0:
                continue

            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            x, y, w, h = cv2.boundingRect(c)

            if area > area_max:
                area_max = area
                cx_c, cy_c, w_c, h_c = cx, cy, w, h

        if area_max > 0:
            print(f"  {color:10s} -> área={int(area_max):6d} px | "
                  f"centro=({cx_c},{cy_c}) | tamaño={w_c}x{h_c}")
        else:
            print(f"  {color:10s} -> no detectado")

        if area_max > mejor_area:
            mejor_area = area_max
            mejor_color = color
            mejor_cx, mejor_cy = cx_c, cy_c
            mejor_w, mejor_h = w_c, h_c

    if mejor_color is None:
        print("Ningún color detectado")
        cv2.imwrite("/tmp/captura_fallida.jpg", frame)
        return None

    # Desplazamiento respecto al centro de la imagen
    dx_px = mejor_cx - cx_img
    dy_px = mejor_cy - cy_img

    # Conversión a milímetros (eje Y invertido por orientación de la cámara)
    dx_mm = dx_px * MM_PER_PIXEL
    dy_mm = -dy_px * MM_PER_PIXEL

    print(f"\n========== RESULTADO ==========")
    print(f"Color detectado:    {mejor_color}")
    print(f"Cubo en pixel:      ({mejor_cx}, {mejor_cy})")
    print(f"Tamaño cubo:        {mejor_w} x {mejor_h} px")
    print(f"Desplazamiento px:  dx={dx_px:+d}, dy={dy_px:+d}")
    print(f"Desplazamiento mm:  dx={dx_mm:+.1f}, dy={dy_mm:+.1f}")

    # Dibujar y guardar imagen anotada
    cv2.rectangle(frame,
                  (mejor_cx - mejor_w // 2, mejor_cy - mejor_h // 2),
                  (mejor_cx + mejor_w // 2, mejor_cy + mejor_h // 2),
                  (0, 255, 0), 2)
    cv2.circle(frame, (mejor_cx, mejor_cy), 6, (0, 0, 255), -1)
    cv2.circle(frame, (cx_img, cy_img), 6, (255, 255, 0), -1)
    cv2.putText(frame, mejor_color,
                (mejor_cx - 30, mejor_cy - mejor_h // 2 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.imwrite("/tmp/cubo_detectado.jpg", frame)
    print("Imagen anotada guardada en /tmp/cubo_detectado.jpg")

    return {
        "color": mejor_color,
        "dx_mm": dx_mm,
        "dy_mm": dy_mm,
        "cx_px": mejor_cx,
        "cy_px": mejor_cy,
        "ancho_cubo": mejor_w,
        "alto_cubo": mejor_h,
    }

print("Función detectar_color_y_medir() cargada")

## 5. Pipeline Completo

La función `ejecutar_pipeline()` integra los 9 pasos del flujo:

1. Mover a posición inicial
2. Detectar color y posición del cubo
3. Ajustar X/Y para alinearse
4. Bajar a agarrar
5. Cerrar gripper
6. Subir con el cubo
7. Llevar a la zona del color
8. Soltar
9. Volver a inicial


In [ ]:
def ejecutar_pipeline():
    # ---------------- PASO 1: Posición inicial ----------------
    print("\n" + "=" * 60)
    print("PASO 1: Moviendo a posición inicial")
    print("=" * 60)
    mc.send_angles(pose_inicial, VELOCIDAD)
    time.sleep(3)
    print(f"Ángulos actuales: {mc.get_angles()}")
    print(f"Coords actuales:  {mc.get_coords()}")

    # ---------------- PASO 2: Detectar color ----------------
    print("\n" + "=" * 60)
    print("PASO 2: Detectando color y posición del cubo")
    print("=" * 60)
    info = detectar_color_y_medir()
    if info is None:
        print("Abortando — no se detectó cubo")
        return

    color = info["color"]
    dx_mm = info["dx_mm"]
    dy_mm = info["dy_mm"]

    # ---------------- PASO 3: Ajustar X/Y ----------------
    print("\n" + "=" * 60)
    print(f"PASO 3: Ajustando X/Y para alinearse con el cubo")
    print("=" * 60)

    coords_actuales = mc.get_coords()
    if coords_actuales is None:
        coords_actuales = list(COORDS_INICIAL)

    x_nuevo = coords_actuales[0] + dx_mm
    y_nuevo = coords_actuales[1] + dy_mm
    rx, ry, rz = coords_actuales[3], coords_actuales[4], coords_actuales[5]

    coords_arriba_cubo = [x_nuevo, y_nuevo, Z_ALTO, rx, ry, rz]
    print(f"Moviendo a (X={x_nuevo:.1f}, Y={y_nuevo:.1f}, Z={Z_ALTO})")
    mc.send_coords(coords_arriba_cubo, VELOCIDAD, 1)
    time.sleep(4)

    # ---------------- PASO 4: Bajar a agarrar ----------------
    print("\n" + "=" * 60)
    print("PASO 4: Bajando a agarrar el cubo")
    print("=" * 60)
    mc.set_gripper_value(100, 50)
    time.sleep(1)

    coords_agarrar = [x_nuevo, y_nuevo, Z_AGARRE, rx, ry, rz]
    print(f"Bajando a Z={Z_AGARRE} mm")
    mc.send_coords(coords_agarrar, VELOCIDAD, 1)
    time.sleep(4)

    # ---------------- PASO 5: Cerrar gripper ----------------
    print("PASO 5: Cerrando gripper")
    mc.set_gripper_value(0, 50)
    time.sleep(1.5)

    # ---------------- PASO 6: Subir ----------------
    print("PASO 6: Subiendo con el cubo")
    coords_subir = [x_nuevo, y_nuevo, Z_ALTO, rx, ry, rz]
    mc.send_coords(coords_subir, VELOCIDAD, 1)
    time.sleep(3)

    # ---------------- PASO 7: Llevar a la zona del color ----------------
    pose_destino = POSICIONES_POR_COLOR[color]
    print("\n" + "=" * 60)
    print(f"PASO 7: Llevando cubo a la ZONA {color}")
    print("=" * 60)
    print(f"Pose destino: {pose_destino}")
    mc.send_angles(pose_destino, VELOCIDAD)
    time.sleep(5)

    # ---------------- PASO 8: Soltar ----------------
    print("PASO 8: Soltando cubo")
    mc.set_gripper_value(100, 50)
    time.sleep(2)

    # ---------------- PASO 9: Volver a inicial ----------------
    print("\n" + "=" * 60)
    print("PASO 9: Volviendo a posición inicial")
    print("=" * 60)
    mc.send_angles(pose_inicial, VELOCIDAD)
    time.sleep(4)

    print(f"\n[OK] PIPELINE COMPLETADO — Cubo {color} depositado en su zona")

print("Función ejecutar_pipeline() cargada")

## 6. Ejecución del Pipeline

Esta celda corre el pipeline completo.
El bloque `try/finally` garantiza que la cámara se libere aunque ocurra un error.


In [ ]:
try:
    ejecutar_pipeline()
finally:
    cap.release()
    print("\nCámara liberada")

## 7. Notas de Calibración

Antes de correr el pipeline en el robot real, ajustar los siguientes parámetros:

### `MM_PER_PIXEL`
- Poner una regla sobre la mesa visible desde la cámara.
- Capturar una imagen y medir cuántos píxeles cubren 100 mm.
- `MM_PER_PIXEL = 100 / píxeles_medidos`.

### `Z_AGARRE`
- Mover manualmente el robot hasta que el gripper toque el cubo.
- Leer `mc.get_coords()` y usar ese valor de Z.

### Signo de `dx_mm` y `dy_mm`
Depende de la orientación física de la cámara respecto al robot:

- Si el cubo está visualmente "a la derecha" y el robot debe moverse a +X, dejar `dx_mm` con signo positivo.
- Si el robot se mueve al lado contrario, invertir el signo.
- Lo mismo aplica para `dy_mm`.

### Rangos HSV
- Si confunde rojo con morado, subir el límite inferior de saturación.
- Si confunde verde con amarillo, ajustar los límites del rango de tono.
- Después de cada corrida abrir `/tmp/cubo_detectado.jpg` para verificar.
